# 2. Feature Engineering — Fraud Prevention Challenge

Este notebook construye el dataset de features listo para modelar, aplicando las decisiones
que surgieron del análisis exploratorio ([`1_EDA.ipynb`](1_EDA.ipynb)) y siguiendo el plan de
trabajo definido en [`0_getting_started.ipynb`](0_getting_started.ipynb) (pasos 2 y 3:
limpieza/feature engineering y split temporal).

**Principio rector: evitar data leakage.** Todo estadístico usado para imputar o codificar
(medianas, frecuencias, categorías top) se calcula **únicamente con el set de entrenamiento**
y luego se aplica tal cual a validación y test. Por eso el split temporal se hace primero, y
recién después se "fitea" cualquier transformación — replicando cómo funcionaría en
producción, donde el modelo nunca ve datos futuros al momento de calcular esos estadísticos.

Estructura de este notebook:
1. Carga de datos
2. Split temporal (train / val / test)
3. Resumen de decisiones de feature engineering
4. Funciones de fit / transform
5. Aplicación del feature engineering a cada split
6. Control de calidad (nulos, columnas, distribución por split)
7. Guardado de datasets procesados y de los parámetros de preprocesamiento
8. Conclusión y próximos pasos

In [1]:
import numpy as np
import pandas as pd
import joblib

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

DATA_PATH = "../data/MercadoLibre Data Scientist Technical Challenge - Dataset.csv"
PROCESSED_DIR = "../data/processed"
ARTIFACTS_DIR = "../artifacts"


## 1. Carga de datos

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["fecha"])
print(f"Shape: {df.shape}")
df.head(3)


Shape: (150000, 19)


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude
0,4,0.6812,50084.12,50.0,0.000000,20.0,AR,1,cat_d26ab52,0.365475,2479.0,952.0,1,NaN,Y,2020-03-20 09:28:19,57.63,100,0
1,4,0.6694,66005.49,0.0,0.000000,2.0,AR,1,cat_ea962fb,0.612728,2603.0,105.0,1,Y,Y,2020-03-09 13:58:28,40.19,25,0
2,4,0.4718,7059.05,4.0,0.463488,92.0,BR,25,cat_4c2544e,0.651835,2153.0,249.0,1,Y,Y,2020-04-08 12:25:55,5.77,23,0


## 2. Split temporal (train / val / test)

Como se vio en el EDA (Sección 11), el dataset cubre 45 días. Usamos un **split temporal**
(70% / 15% / 15%, ordenado por `fecha`) en vez de uno aleatorio: así el set de validación y
el de test siempre contienen transacciones *posteriores* a las de entrenamiento, igual que en
producción, donde el modelo solo puede aprender del pasado para predecir el futuro. Un split
aleatorio sería optimista: dejaría "fugar" información del futuro hacia el entrenamiento
(por ejemplo, la frecuencia de una categoría `j` calculada con datos que en producción todavía
no existirían).

In [3]:
def temporal_split(data: pd.DataFrame, val_frac: float = 0.15, test_frac: float = 0.15):
    data_sorted = data.sort_values("fecha").reset_index(drop=True)
    n = len(data_sorted)
    train_end = int(n * (1 - val_frac - test_frac))
    val_end = int(n * (1 - test_frac))
    return (
        data_sorted.iloc[:train_end].copy(),
        data_sorted.iloc[train_end:val_end].copy(),
        data_sorted.iloc[val_end:].copy(),
    )


train_df, val_df, test_df = temporal_split(df)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(
        f"{name:5s}: {len(split):>7,} filas | "
        f"{split['fecha'].min()} -> {split['fecha'].max()} | "
        f"tasa fraude: {split['fraude'].mean():.4f}"
    )


train: 105,000 filas | 2020-03-08 00:02:15 -> 2020-04-10 00:12:50 | tasa fraude: 0.0518
val  :  22,500 filas | 2020-04-10 00:12:58 -> 2020-04-16 14:50:10 | tasa fraude: 0.0453
test :  22,500 filas | 2020-04-16 14:51:15 -> 2020-04-21 23:59:56 | tasa fraude: 0.0460


Los tres rangos de fecha son consecutivos y no se superponen, y la tasa de fraude es
similar entre splits (~5%), consistente con lo observado en el EDA (sin drift fuerte día a
día). Esto da confianza en que el split es representativo y no introduce un desbalance
adicional entre train/val/test.

## 3. Resumen de decisiones de feature engineering

Basado en los hallazgos del EDA, estas son las transformaciones aplicadas a cada columna:

| Columna(s) | Transformación | Motivo (evidencia del EDA) |
|---|---|---|
| `a` | One-hot (drop primer nivel) | Categórica de 4 niveles con tasas de fraude distintas |
| `b`, `c` | Imputación con mediana de train + flag único `bc_is_null` | Nulos exactamente coincidentes (100% de solapamiento) → probablemente del mismo proceso de enriquecimiento |
| `c` | `log1p` | Escala muy amplia (hasta ~1.39M), fuerte asimetría |
| `d` | Imputación con mediana + flag `d_is_capped` (`d==50`) | 25.7% de las filas están exactamente en el valor capado (50) — señal en sí misma |
| `e` | `log1p` | Fuertemente sesgada a la derecha (máx. 833 vs. mediana 0.10) |
| `f` | Imputación con mediana (sin `log1p`) | Tiene valores negativos (mín. -5) → `log1p` no es válido |
| `g` | Colapsar a top-10 países + "OTHER" + "MISSING" (nulos), luego one-hot | 51 categorías, muy concentradas en pocos países; nulos deben tratarse aparte |
| `h` | Sin cambios | Sin nulos, ya numérica |
| `j` | Frequency encoding (frecuencia relativa en train) | 8,324 categorías → one-hot inviable; se evita target encoding para no arriesgar leakage/overfitting en categorías de bajo volumen |
| `k` | **Se descarta** (no se incluye como feature) | Es un identificador único: sus 150,000 valores son todos distintos (ver `1_EDA.ipynb`, Sección 2) — no aporta señal de negocio y arriesga que el modelo memorice en vez de generalizar |
| `l`, `m` | Imputación con mediana + `log1p` | Nulos mínimos; distribuciones sesgadas |
| `n` | Sin cambios (ya 0/1) | — |
| `o` | Flags `o_is_null`, `o_is_Y`, `o_is_N` | 72.6% de nulos y **no aleatorios** (MNAR): la ausencia es en sí misma predictiva |
| `p` | Mapeo a 0/1 (`Y`→1) | Binaria sin nulos |
| `score`, `monto` | Sin cambios + `monto_log` (`log1p`) adicional | `score` ya está en escala 0-100; `monto` es muy sesgado |
| `fecha` | Derivar `hora`, `es_madrugada` (hora 0-5), `dia_semana` | Patrón horario fuerte (fraude ~11.3% de madrugada vs. ~4.7% el resto); día de la semana sin patrón claro pero se deja como feature numérica |

Todos los estadísticos (medianas, top países, frecuencias de `j`) se calculan **solo con
`train_df`** en la Sección 4 y se reutilizan sin cambios para `val_df` y `test_df`.

## 4. Funciones de fit / transform

In [4]:
def fit_preprocessing(train: pd.DataFrame, top_n_countries: int = 10) -> dict:
    """Calcula, a partir de train, todos los estadísticos necesarios para transformar."""
    params = {
        "median_b": train["b"].median(),
        "median_c": train["c"].median(),
        "median_d": train["d"].median(),
        "median_f": train["f"].median(),
        "median_l": train["l"].median(),
        "median_m": train["m"].median(),
        "top_countries": train["g"].value_counts().head(top_n_countries).index.tolist(),
    }
    j_counts = train["j"].value_counts()
    params["j_freq_map"] = (j_counts / len(train)).to_dict()
    params["j_freq_default"] = 0.0  # categorías de j nunca vistas en train
    return params


In [5]:
def transform(data: pd.DataFrame, params: dict) -> pd.DataFrame:
    """Aplica el feature engineering usando SOLO estadísticos ya calculados en params."""
    out = pd.DataFrame(index=data.index)

    # a: one-hot
    out = pd.concat([out, pd.get_dummies(data["a"], prefix="a", drop_first=True).astype(int)], axis=1)

    # b, c: mismo patrón de nulos -> un flag compartido
    out["bc_is_null"] = data["b"].isna().astype(int)
    out["b"] = data["b"].fillna(params["median_b"])
    out["c_log"] = np.log1p(data["c"].fillna(params["median_c"]))

    # d: imputación + flag de valor capado
    out["d"] = data["d"].fillna(params["median_d"])
    out["d_is_capped"] = (data["d"] == 50).astype(int)

    # e: sesgada, sin nulos
    out["e_log"] = np.log1p(data["e"])

    # f: tiene negativos -> no se puede aplicar log1p
    out["f"] = data["f"].fillna(params["median_f"])

    # g: colapsar categorías raras/ausentes y one-hot
    g_clean = data["g"]
    is_top_or_na = g_clean.isin(params["top_countries"]) | g_clean.isna()
    g_clean = g_clean.where(is_top_or_na, other="OTHER").fillna("MISSING")
    out = pd.concat([out, pd.get_dummies(g_clean, prefix="g").astype(int)], axis=1)

    # h: sin cambios. k NO se incluye: es un identificador único (ver 1_EDA.ipynb, Sección 2)
    out["h"] = data["h"]

    # j: frequency encoding ajustado en train
    out["j_freq"] = data["j"].map(params["j_freq_map"]).fillna(params["j_freq_default"])

    # l, m: imputación + log
    out["l_log"] = np.log1p(data["l"].fillna(params["median_l"]))
    out["m_log"] = np.log1p(data["m"].fillna(params["median_m"]))

    # n, p: binarias
    out["n"] = data["n"]
    out["p"] = (data["p"] == "Y").astype(int)

    # o: flags (is_null / is_Y / is_N) en vez de imputar
    out["o_is_null"] = data["o"].isna().astype(int)
    out["o_is_Y"] = (data["o"] == "Y").astype(int)
    out["o_is_N"] = (data["o"] == "N").astype(int)

    # score, monto
    out["score"] = data["score"]
    out["monto"] = data["monto"]
    out["monto_log"] = np.log1p(data["monto"])

    # features temporales
    out["hora"] = data["fecha"].dt.hour
    out["es_madrugada"] = out["hora"].between(0, 5).astype(int)
    out["dia_semana"] = data["fecha"].dt.dayofweek

    return out


## 5. Aplicación del feature engineering a cada split

`fit_preprocessing` se ejecuta **solo sobre `train_df`**. Las columnas de `X_val` y `X_test`
se alinean explícitamente a las de `X_train` (`reindex(..., fill_value=0)`): así, si val o
test tuvieran una categoría de `g` que no exista en train, o si algún nivel de `a` faltara en
un split, el resultado sigue siendo consistente en vez de romper o generar columnas
desalineadas — el mismo tipo de problema que causaría un error silencioso en producción.

In [6]:
params = fit_preprocessing(train_df)

X_train = transform(train_df, params)
X_val = transform(val_df, params).reindex(columns=X_train.columns, fill_value=0)
X_test = transform(test_df, params).reindex(columns=X_train.columns, fill_value=0)

y_train, y_val, y_test = train_df["fraude"], val_df["fraude"], test_df["fraude"]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
X_train.head(3)


X_train: (105000, 37) | X_val: (22500, 37) | X_test: (22500, 37)


,a_2,a_3,a_4,bc_is_null,b,c_log,d,d_is_capped,e_log,f,g_AR,g_BR,g_CO,g_ES,g_GB,g_MISSING,g_MX,g_OTHER,g_RU,g_SE,g_US,g_UY,h,j_freq,l_log,m_log,n,p,o_is_null,o_is_Y,o_is_N,score,monto,monto_log,hora,es_madrugada,dia_semana
0,0,0,1,0,0.7388,8.750762,14.0,0,0.130395,24.0,0,1,0,0,0,0,0,0,0,0,0,0,7,0.004562,7.767264,6.093570,1,1,1,0,0,25,22.18,3.143290,0,1,6
1,0,0,1,0,0.7548,9.960439,20.0,0,0.415293,7.0,0,1,0,0,0,0,0,0,0,0,0,0,2,0.000133,7.751475,4.304065,1,0,1,0,0,7,6.00,1.945910,0,1,6
2,0,0,1,0,0.9026,8.297501,50.0,1,0.242292,1.0,0,1,0,0,0,0,0,0,0,0,0,0,3,0.002648,5.463832,5.451038,1,1,0,0,1,91,26.67,3.320349,0,1,6


## 6. Control de calidad

In [7]:
nulls_after = pd.concat(
    [X_train.isna().sum(), X_val.isna().sum(), X_test.isna().sum()],
    axis=1, keys=["train", "val", "test"],
)
assert nulls_after.values.sum() == 0, "Quedaron nulos sin tratar"
print("Sin nulos remanentes en ninguna feature de ningún split. OK.")


Sin nulos remanentes en ninguna feature de ningún split. OK.


In [8]:
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns)
print(f"Columnas consistentes entre splits: {X_train.shape[1]} features. OK.")
list(X_train.columns)


Columnas consistentes entre splits: 37 features. OK.


['a_2',
 'a_3',
 'a_4',
 'bc_is_null',
 'b',
 'c_log',
 'd',
 'd_is_capped',
 'e_log',
 'f',
 'g_AR',
 'g_BR',
 'g_CO',
 'g_ES',
 'g_GB',
 'g_MISSING',
 'g_MX',
 'g_OTHER',
 'g_RU',
 'g_SE',
 'g_US',
 'g_UY',
 'h',
 'j_freq',
 'l_log',
 'm_log',
 'n',
 'p',
 'o_is_null',
 'o_is_Y',
 'o_is_N',
 'score',
 'monto',
 'monto_log',
 'hora',
 'es_madrugada',
 'dia_semana']

In [9]:
compare_cols = ["score", "j_freq", "monto_log", "hora"]
summary = pd.concat(
    [X_train[compare_cols].mean(), X_val[compare_cols].mean(), X_test[compare_cols].mean()],
    axis=1, keys=["train", "val", "test"],
)
summary


,train,val,test
score,49.520752,44.646089,44.698667
j_freq,0.001898,0.001841,0.001690
monto_log,3.135974,3.040522,3.082101
hora,14.048505,14.029911,14.673378


Las medias de estas features clave se mantienen razonablemente estables entre splits — no
hay señales de un corte temporal problemático (por ejemplo, un cambio brusco en `j_freq`
indicaría que en val/test aparecen categorías muy distintas a las vistas en train). Esta
misma comparación es la base del monitoreo que se recomienda en producción: si estas medias
empiezan a divergir respecto a las de entrenamiento, es una alerta temprana de *data drift*
(ver preguntas 3 y 4 del enunciado, respondidas en el notebook de modelado).

## 7. Guardado de datasets procesados y parámetros de preprocesamiento

In [10]:
import os

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

for name, X, y, raw in [
    ("train", X_train, y_train, train_df),
    ("val", X_val, y_val, val_df),
    ("test", X_test, y_test, test_df),
]:
    out = X.copy()
    out["fraude"] = y.values
    out["fecha"] = raw["fecha"].values  # metadata: no es una feature, útil para análisis posteriores
    out.to_csv(f"{PROCESSED_DIR}/{name}.csv", index=False)
    print(f"Guardado {PROCESSED_DIR}/{name}.csv -> {out.shape}")


Guardado ../data/processed/train.csv -> (105000, 39)


Guardado ../data/processed/val.csv -> (22500, 39)


Guardado ../data/processed/test.csv -> (22500, 39)


In [11]:
joblib.dump(
    {"params": params, "feature_columns": list(X_train.columns)},
    f"{ARTIFACTS_DIR}/preprocessing_params.pkl",
)
print(f"Guardado {ARTIFACTS_DIR}/preprocessing_params.pkl")


Guardado ../artifacts/preprocessing_params.pkl


Guardar los parámetros de preprocesamiento (medianas, top países, frecuencias de `j`) junto
con la lista exacta de columnas de `X_train` permite reutilizar **la misma transformación**
en el notebook de modelado y, eventualmente, en producción — aplicando siempre `transform()`
con los estadísticos congelados de train, nunca recalculándolos sobre datos nuevos.

## 8. Conclusión y próximos pasos

- Se aplicó un **split temporal** 70/15/15 (train/val/test), sin overlap de fechas, con tasa
  de fraude estable entre splits (~5%).
- Todas las imputaciones y encodings se ajustaron **exclusivamente con train**, evitando
  leakage hacia val/test — una práctica que además responde directamente a la pregunta del
  enunciado sobre cómo lograr que la performance en producción se parezca a la de laboratorio.
- El dataset final expande las 17 columnas originales en un conjunto más amplio de features
  numéricas y dummies (una por cada `a`/`g` codificada, más las columnas numéricas/flags),
  sin nulos, con las columnas de `X_val`/`X_test` alineadas explícitamente a las de
  `X_train` (ver el shape impreso en la Sección 5).
- Se guardaron `train.csv`, `val.csv` y `test.csv` en `data/processed/`, y los parámetros de
  preprocesamiento en `artifacts/preprocessing_params.pkl` para reutilizar en el modelado.

**Próximo paso:** notebook `3_modeling` — entrenar Regresión Logística, Random Forest,
XGBoost y LightGBM sobre `X_train`/`y_train`, ajustando el umbral de decisión con la
ecuación de beneficio de `0_getting_started.ipynb` sobre `X_val`, y evaluando la performance
final (ROC-AUC, PR-AUC, F1, F6 y beneficio total) sobre `X_test`.